In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)
print(documents[2].keys())

dict_keys(['content', 'filename'])


In [12]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]

)
index.fit(documents)

In [21]:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(query, num_results=1)

top_filename = search_results[0]["filename"]
print("result is: ", top_filename)

result is:  01-agentic-rag/lessons/14-agentic-loop.md


In [26]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [33]:
from openai import OpenAI
from rag_helper import RAGBase

class CourseRAG(RAGBase):
    
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response 

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        
        response = self.llm(prompt)
        
        answer = response.output_text
        input_tokens = response.usage.input_tokens
        
        return answer, input_tokens

openai_client = OpenAI()

rag_assistant = CourseRAG(
    index=index, 
    llm_client=openai_client
)

query = "How does the agentic loop keep calling the model until it stops?"
answer, tokens = rag_assistant.rag(query)

print(f"Input tokens sent: {tokens}")

Input tokens sent: 7126


In [36]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Number of chunks created: ", len(chunks))
print("/n Keys in the chuks: ", chunks[0].keys())

Number of chunks created:  295
/n Keys in the chuks:  dict_keys(['start', 'content', 'filename'])


In [38]:
chunk_index = minsearch.Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

chunk_rag_assistant = CourseRAG(
    index = chunk_index,
    llm_client=openai_client
)

query = "How does the agentic loop keep calling the model until it stops?"

chunk_answer, chunk_tokens = chunk_rag_assistant.rag(query)
print(f"Input tokens sent with chunking: {chunk_tokens}")

Input tokens sent with chunking: 2309


In [41]:
def search(query: str) -> list[dict]:
    """
        Search the course lessons for relevant content based on the user query.
    """
    return chunk_index.search(query, num_results=5)


In [43]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """ 
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt = instructions,
    chat_interface = chat_interface,
    llm_client = OpenAIClient()
)
prompt = "How does the agentic loop work, and how is it different from plain RAG?"
result = runner.loop(prompt=prompt, callback=callback)

-> Response received


-> Response received
